# 06 — Encoding Model: All Conditions

Runs Ridge Regression encoding models for **all conditions** across **all embedding
modes** and saves results to disk. All plots and visualization is in notebook 07.

## Modes

| Mode             | Source  | Model        | Context                        |
|------------------|---------|--------------|--------------------------------|
| `contextual`     | NB 01–03| XLM-RoBERTa  | Full sentence, intersection    |
| `sliding_window` | NB 04   | XLM-RoBERTa  | ±8-word window                 |
| `xglm`           | NB 04b  | XGLM-1.7B    | Full transcript, causal        |
| `fasttext`       | Eyal    | FastText     | Static (no context)            |

## Conditions per mode

Each mode runs: EN alone, HE alone, AR alone, EN+HE residual,
EN+AR residual, and noise baseline.

## Electrode selection

Per subject: electrodes where peak r in the **English condition of the
sliding_window mode** exceeds the threshold. Saved to disk so notebook 07
can use the same selection consistently.

## Output structure

```
results/
  encoding_{mode}_64Hz_(-2.0,2.0)/
    corrs subj={subj} - {condition}.npy   # (n_folds, n_electrodes, n_timepoints)
  selected_electrodes.json                # {subj: [electrode indices]}
```

## 1. Imports

In [ ]:
import sys
import os
import json
import io
import warnings
from contextlib import redirect_stdout

sys.path.insert(0, os.path.abspath('../data/Amirim_Project_Submission'))

import numpy as np
import pandas as pd
from tqdm import tqdm
import mne
from mne_bids import BIDSPath
from static_encoding import process_embeddings

print('Imports ready.')

## 2. Configuration

In [ ]:
DATA_DIR        = '../data/processed/'
WORD_LEVEL_PATH = '../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv'
FT_PATH         = '../data/Amirim_Project_Submission/podcast_trilingual_embeddings.csv'
RESULTS_DIR     = '../results/'

freq       = 64
tmin, tmax = -2.0, 2.0
use_PCA    = True
PCA_dim    = 150
subjects   = [f'{i:02d}' for i in range(1, 10)]

# Electrode selection — applied after encoding
# 'threshold' : keep electrodes with peak r >= THRESHOLD
# 'percentile': keep top (100-PERCENTILE)% of electrodes
SELECTION_MODE = 'threshold'
THRESHOLD      = 0.10
PERCENTILE     = 75

# Reference condition for electrode selection
SELECTION_REF_MODE = 'contextual'
SELECTION_REF_COND = 'en_ctx'

os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Subjects         : {subjects}')
print(f'Freq / tmin-tmax : {freq}Hz / ({tmin}, {tmax})')
print(f'PCA              : {use_PCA} -> {PCA_dim}d')
print(f'Electrode sel    : {SELECTION_MODE} '
      f'(ref: {SELECTION_REF_MODE}/{SELECTION_REF_COND})')

## 3. Load All Embeddings

In [ ]:
full_transcript = pd.read_csv(WORD_LEVEL_PATH)
word_df_all     = full_transcript[['start', 'end']].reset_index(drop=True)

def parse_embedding(x):
    if isinstance(x, str):
        x = x.replace('[', '').replace(']', '')
        return np.array(x.split(), dtype=np.float32)
    return np.array(x, dtype=np.float32)


# ── Helper: load a CSV embedding matrix ──────────────────────────────────────
def try_load_csv(path):
    if not os.path.exists(path):
        return None
    return pd.read_csv(path).values.astype(np.float32)

def try_load_npy(path):
    if not os.path.exists(path):
        return None
    return np.load(path).astype(np.float32)


# ── FastText ──────────────────────────────────────────────────────────────────
ft_available = os.path.exists(FT_PATH)
if ft_available:
    ft_df    = pd.read_csv(FT_PATH)
    E_ft     = np.vstack(ft_df['en_embedding'].apply(parse_embedding).values)
    H_ft     = np.vstack(ft_df['he_embedding'].apply(parse_embedding).values)
    A_ft     = np.vstack(ft_df['ar_embedding'].apply(parse_embedding).values)
    H_ft_res = try_load_npy(DATA_DIR + 'fasttext_hebrew_residuals.npy')
    A_ft_res = try_load_npy(DATA_DIR + 'fasttext_arabic_residuals.npy')
    print(f'FastText loaded : EN{E_ft.shape} HE{H_ft.shape} AR{A_ft.shape}')
else:
    print('FastText not found — fasttext mode will be skipped.')

# ── Contextual XLM-RoBERTa (sentence-level, intersection) ────────────────────
ctx_available = all(os.path.exists(DATA_DIR + f) for f in [
    'contextual_shared_indices.csv',
    'en_contextual_aligned_embeddings.csv',
    'he_contextual_aligned_embeddings.csv',
    'ar_contextual_aligned_embeddings.csv',
])
if ctx_available:
    shared_idx = pd.read_csv(DATA_DIR + 'contextual_shared_indices.csv')['original_word_idx'].values
    en_pos = {v: i for i, v in enumerate(pd.read_csv(DATA_DIR + 'en_contextual_matched_indices.csv')['original_word_idx'].values)}
    he_pos = {v: i for i, v in enumerate(pd.read_csv(DATA_DIR + 'he_contextual_matched_indices.csv')['original_word_idx'].values)}
    ar_pos = {v: i for i, v in enumerate(pd.read_csv(DATA_DIR + 'ar_contextual_matched_indices.csv')['original_word_idx'].values)}
    E_ctx  = pd.read_csv(DATA_DIR + 'en_contextual_aligned_embeddings.csv').values.astype(np.float32)[[en_pos[i] for i in shared_idx]]
    H_ctx  = pd.read_csv(DATA_DIR + 'he_contextual_aligned_embeddings.csv').values.astype(np.float32)[[he_pos[i] for i in shared_idx]]
    A_ctx  = pd.read_csv(DATA_DIR + 'ar_contextual_aligned_embeddings.csv').values.astype(np.float32)[[ar_pos[i] for i in shared_idx]]
    H_ctx_res = try_load_npy(DATA_DIR + 'hebrew_residuals_contextual.npy')
    A_ctx_res = try_load_npy(DATA_DIR + 'arabic_residuals_contextual.npy')
    ctx_word_df = full_transcript.iloc[shared_idx][['start', 'end']].reset_index(drop=True)
    print(f'Contextual (ctx) loaded : EN{E_ctx.shape} HE{H_ctx.shape} AR{A_ctx.shape}')
else:
    print('Contextual embeddings not found — contextual mode will be skipped.')

# ── Sliding window XLM-RoBERTa ────────────────────────────────────────────────
sw_available = all(os.path.exists(DATA_DIR + f) for f in [
    'en_sliding_window_embeddings.csv',
    'he_sliding_window_embeddings.csv',
    'ar_sliding_window_embeddings.csv',
])
if sw_available:
    E_sw     = try_load_csv(DATA_DIR + 'en_sliding_window_embeddings.csv')
    H_sw     = try_load_csv(DATA_DIR + 'he_sliding_window_embeddings.csv')
    A_sw     = try_load_csv(DATA_DIR + 'ar_sliding_window_embeddings.csv')
    H_sw_res = try_load_npy(DATA_DIR + 'hebrew_residuals_sliding_window.npy')
    A_sw_res = try_load_npy(DATA_DIR + 'arabic_residuals_sliding_window.npy')
    print(f'Sliding window loaded   : EN{E_sw.shape} HE{H_sw.shape} AR{A_sw.shape}')
else:
    print('Sliding window embeddings not found — sliding_window mode will be skipped.')

# ── XGLM ─────────────────────────────────────────────────────────────────────
xglm_available = all(os.path.exists(DATA_DIR + f) for f in [
    'en_xglm_embeddings.csv',
    'he_xglm_embeddings.csv',
    'ar_xglm_embeddings.csv',
])
if xglm_available:
    E_xglm     = try_load_csv(DATA_DIR + 'en_xglm_embeddings.csv')
    H_xglm     = try_load_csv(DATA_DIR + 'he_xglm_embeddings.csv')
    A_xglm     = try_load_csv(DATA_DIR + 'ar_xglm_embeddings.csv')
    H_xglm_res = try_load_npy(DATA_DIR + 'hebrew_residuals_xglm.npy')
    A_xglm_res = try_load_npy(DATA_DIR + 'arabic_residuals_xglm.npy')
    print(f'XGLM loaded             : EN{E_xglm.shape} HE{H_xglm.shape} AR{A_xglm.shape}')
else:
    print('XGLM embeddings not found — xglm mode will be skipped.')

## 4. Define All Conditions

In [ ]:
# Each condition: (name, language_mode, en_mat, he_mat, ar_mat, word_df, out_folder)
# Modes whose embeddings are unavailable are excluded automatically.

def make_noise(mat):
    """Gaussian noise matched to the shape and scale of mat."""
    return np.random.normal(mat.mean(), mat.std(), mat.shape).astype(np.float32)


MODES_CONDITIONS = []

# ── FastText (static, no context) ─────────────────────────────────────────────
if ft_available:
    out = os.path.join(RESULTS_DIR, f'encoding_fasttext_{freq}Hz_({tmin},{tmax})')
    os.makedirs(out, exist_ok=True)
    ft_res_he = H_ft_res if H_ft_res is not None else make_noise(H_ft)
    ft_res_ar = A_ft_res if A_ft_res is not None else make_noise(A_ft)
    MODES_CONDITIONS += [
        ('en_ft',          'en',    E_ft, H_ft,      A_ft,      word_df_all, out),
        ('he_ft',          'he',    E_ft, H_ft,      A_ft,      word_df_all, out),
        ('ar_ft',          'ar',    E_ft, H_ft,      A_ft,      word_df_all, out),
        ('en+he_ft_res',   'en+he', E_ft, ft_res_he, A_ft,      word_df_all, out),
        ('en+ar_ft_res',   'en+ar', E_ft, H_ft,      ft_res_ar, word_df_all, out),
        ('noise_ft',       'noise', E_ft, H_ft,      A_ft,      word_df_all, out),
    ]
    print(f'FastText        : {len([c for c in MODES_CONDITIONS if c[6]==out])} conditions')

# ── Contextual XLM-RoBERTa (sentence-level) ───────────────────────────────────
if ctx_available:
    out = os.path.join(RESULTS_DIR, f'encoding_contextual_{freq}Hz_({tmin},{tmax})')
    os.makedirs(out, exist_ok=True)
    ctx_res_he = H_ctx_res if H_ctx_res is not None else make_noise(H_ctx)
    ctx_res_ar = A_ctx_res if A_ctx_res is not None else make_noise(A_ctx)
    MODES_CONDITIONS += [
        ('en_ctx',         'en',    E_ctx, H_ctx,      A_ctx,      ctx_word_df, out),
        ('he_ctx',         'he',    E_ctx, H_ctx,      A_ctx,      ctx_word_df, out),
        ('ar_ctx',         'ar',    E_ctx, H_ctx,      A_ctx,      ctx_word_df, out),
        ('en+he_ctx_res',  'en+he', E_ctx, ctx_res_he, A_ctx,      ctx_word_df, out),
        ('en+ar_ctx_res',  'en+ar', E_ctx, H_ctx,      ctx_res_ar, ctx_word_df, out),
        ('noise_ctx',      'noise', E_ctx, H_ctx,      A_ctx,      ctx_word_df, out),
    ]
    print(f'Contextual      : {len([c for c in MODES_CONDITIONS if c[6]==out])} conditions')

# ── Sliding window XLM-RoBERTa ────────────────────────────────────────────────
if sw_available:
    out = os.path.join(RESULTS_DIR, f'encoding_sliding_window_{freq}Hz_({tmin},{tmax})')
    os.makedirs(out, exist_ok=True)
    sw_res_he = H_sw_res if H_sw_res is not None else make_noise(H_sw)
    sw_res_ar = A_sw_res if A_sw_res is not None else make_noise(A_sw)
    MODES_CONDITIONS += [
        ('en_sw',          'en',    E_sw, H_sw,      A_sw,      word_df_all, out),
        ('he_sw',          'he',    E_sw, H_sw,      A_sw,      word_df_all, out),
        ('ar_sw',          'ar',    E_sw, H_sw,      A_sw,      word_df_all, out),
        ('en+he_sw_res',   'en+he', E_sw, sw_res_he, A_sw,      word_df_all, out),
        ('en+ar_sw_res',   'en+ar', E_sw, H_sw,      sw_res_ar, word_df_all, out),
        ('noise_sw',       'noise', E_sw, H_sw,      A_sw,      word_df_all, out),
    ]
    print(f'Sliding window  : {len([c for c in MODES_CONDITIONS if c[6]==out])} conditions')

# ── XGLM (full causal context) ────────────────────────────────────────────────
if xglm_available:
    out = os.path.join(RESULTS_DIR, f'encoding_xglm_{freq}Hz_({tmin},{tmax})')
    os.makedirs(out, exist_ok=True)
    xglm_res_he = H_xglm_res if H_xglm_res is not None else make_noise(H_xglm)
    xglm_res_ar = A_xglm_res if A_xglm_res is not None else make_noise(A_xglm)
    MODES_CONDITIONS += [
        ('en_xglm',        'en',    E_xglm, H_xglm,       A_xglm,       word_df_all, out),
        ('he_xglm',        'he',    E_xglm, H_xglm,       A_xglm,       word_df_all, out),
        ('ar_xglm',        'ar',    E_xglm, H_xglm,       A_xglm,       word_df_all, out),
        ('en+he_xglm_res', 'en+he', E_xglm, xglm_res_he,  A_xglm,       word_df_all, out),
        ('en+ar_xglm_res', 'en+ar', E_xglm, H_xglm,       xglm_res_ar,  word_df_all, out),
        ('noise_xglm',     'noise', E_xglm, H_xglm,       A_xglm,       word_df_all, out),
    ]
    print(f'XGLM            : {len([c for c in MODES_CONDITIONS if c[6]==out])} conditions')

print(f'\nTotal conditions : {len(MODES_CONDITIONS)}')
print(f'Total runs       : {len(subjects)} x {len(MODES_CONDITIONS)} = '
      f'{len(subjects) * len(MODES_CONDITIONS)}')
print()
print(f'{"Name":22s}  {"Mode":8s}  {"Words":>6s}  {"Dim":>6s}')
print('-' * 50)
for name, mode, en, he, ar, wdf, folder in MODES_CONDITIONS:
    print(f'{name:22s}  {mode:8s}  {len(wdf):>6d}  {en.shape[1]:>6d}d')

## 5. Encoding Loop

In [ ]:
for subj in subjects:
    file_path = BIDSPath(
        root='../data/ds005574/derivatives/ecogprep',
        subject=subj, task='podcast', datatype='ieeg',
        description='highgamma', suffix='ieeg', extension='.fif'
    ).fpath
    fif_path = str(file_path)

    if not os.path.exists(fif_path):
        print(f'Subject {subj}: .fif not found, skipping.')
        continue

    pending = [
        c for c in MODES_CONDITIONS
        if not os.path.exists(os.path.join(c[6], f'corrs subj={subj} - {c[0]}.npy'))
    ]
    if not pending:
        print(f'Subject {subj}: all {len(MODES_CONDITIONS)} conditions already saved.')
        continue

    raw = mne.io.read_raw_fif(fif_path, verbose=False)
    print(f'\nSubject {subj}: {len(raw.info["ch_names"])} channels | '
          f'{len(pending)}/{len(MODES_CONDITIONS)} to run')

    for name, mode, en_mat, he_mat, ar_mat, base_df, outpath in tqdm(pending, desc=f'Subj {subj}'):
        embedding_df = base_df.copy()
        embedding_df['en_embedding'] = list(en_mat.astype(np.float32))
        embedding_df['he_embedding'] = list(he_mat.astype(np.float32))
        embedding_df['ar_embedding'] = list(ar_mat.astype(np.float32))

        f = io.StringIO()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            with redirect_stdout(f):
                _, cv_scores = process_embeddings(
                    embedding_df=embedding_df, raw=raw,
                    channel_names_regex='', freq=freq,
                    tmin=tmin, tmax=tmax, language_mode=mode,
                    random_noise_mode='over all embeds',
                    use_PCA=use_PCA, PCA_dim=PCA_dim
                )
        np.save(os.path.join(outpath, f'corrs subj={subj} - {name}.npy'), cv_scores)

print('\nEncoding loop complete.')

## 6. Electrode Selection

Select language-responsive electrodes per subject based on the reference
condition. Save selection to disk so notebook 07 uses the same electrodes.

In [ ]:
# Reference output folder for electrode selection
ref_folder_map = {
    'sliding_window': os.path.join(RESULTS_DIR, f'encoding_sliding_window_{freq}Hz_({tmin},{tmax})'),
    'contextual'    : os.path.join(RESULTS_DIR, f'encoding_contextual_{freq}Hz_({tmin},{tmax})'),
    'xglm'          : os.path.join(RESULTS_DIR, f'encoding_xglm_{freq}Hz_({tmin},{tmax})'),
    'fasttext'      : os.path.join(RESULTS_DIR, f'encoding_fasttext_{freq}Hz_({tmin},{tmax})'),
}
ref_folder = ref_folder_map[SELECTION_REF_MODE]

selected_electrodes = {}   # subj -> list of electrode indices

sel_tag = (f'top{100-PERCENTILE}pct' if SELECTION_MODE == 'percentile'
           else f'r{str(THRESHOLD).replace(".","p")}')

print(f'Electrode selection ref  : {SELECTION_REF_MODE} / {SELECTION_REF_COND}')
print(f'Selection mode           : {SELECTION_MODE} ({sel_tag})')
print()
print(f'{"Subj":>6s}  {"Total":>8s}  {"Selected":>10s}  {"% sel":>8s}  {"ref r":>8s}')
print('-' * 48)

for subj in subjects:
    ref_fname = os.path.join(ref_folder, f'corrs subj={subj} - {SELECTION_REF_COND}.npy')
    if not os.path.exists(ref_fname):
        print(f'{subj:>6s}  reference file not found, skipping.')
        continue

    scores = np.load(ref_fname)           # (folds, electrodes, timepoints)
    peak_r = scores.mean(0).max(-1)       # (electrodes,)

    if SELECTION_MODE == 'percentile':
        thresh = np.percentile(peak_r, PERCENTILE)
    else:
        thresh = THRESHOLD

    selected = np.where(peak_r >= thresh)[0].tolist()
    selected_electrodes[subj] = selected

    print(f'{subj:>6s}  {len(peak_r):>8d}  {len(selected):>10d}  '
          f'{len(selected)/len(peak_r)*100:>7.1f}%  {thresh:>8.4f}')

# Save to disk for notebook 07
sel_path = os.path.join(RESULTS_DIR, f'selected_electrodes_{sel_tag}.json')
with open(sel_path, 'w') as f:
    json.dump({k: v for k, v in selected_electrodes.items()}, f, indent=2)
print(f'\nSaved: {sel_path}')

## 7. Summary Table

Mean peak r across subjects, computed only on selected electrodes.
This gives a quick sanity check that the encoding ran correctly.
Full analysis and plots are in notebook 07.

In [ ]:
def mean_peak_selected(folder, cond_name):
    peaks = []
    for subj in subjects:
        fname = os.path.join(folder, f'corrs subj={subj} - {cond_name}.npy')
        if not os.path.exists(fname) or subj not in selected_electrodes:
            continue
        scores = np.load(fname)
        sel    = selected_electrodes[subj]
        if len(sel) == 0:
            continue
        peak = scores.mean(0)[sel].max(-1).mean()
        peaks.append(peak)
    return np.mean(peaks) if peaks else float('nan')


# One row per (mode, condition-pair)
SUMMARY_ROWS = []
for mode_name, ref_folder_key in [
    ('fasttext',      'fasttext'),
    ('contextual',    'contextual'),
    ('sliding_window','sliding_window'),
    ('xglm',          'xglm'),
]:
    folder = ref_folder_map[mode_name]
    suffix = {'fasttext': 'ft', 'contextual': 'ctx',
              'sliding_window': 'sw', 'xglm': 'xglm'}[mode_name]
    for cond in [f'en_{suffix}', f'he_{suffix}', f'ar_{suffix}',
                 f'en+he_{suffix}_res', f'en+ar_{suffix}_res', f'noise_{suffix}']:
        r = mean_peak_selected(folder, cond)
        SUMMARY_ROWS.append({'mode': mode_name, 'condition': cond, 'mean_peak_r': r})

summary_df = pd.DataFrame(SUMMARY_ROWS)
summary_df.to_csv(os.path.join(RESULTS_DIR, 'encoding_summary.csv'), index=False)

print('=' * 55)
print('ENCODING SUMMARY — selected electrodes')
print('=' * 55)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))
print()
print(f'Full results saved to: {os.path.abspath(RESULTS_DIR)}')
print('Proceed to notebook 07 for all plots and statistical tests.')